# 3D Gaussian Splatting: Photographs to a 3D Scene

This notebook documents an end-to-end **3D Gaussian Splatting (3DGS)** experiment in Google Colab. It prepares a set of photographs, estimates camera poses and sparse geometry with COLMAP, trains a Gaussian scene representation, and writes a `.ply` output.

> **Environment note:** 3DGS depends on compiled CUDA extensions. The original experiment used legacy Python/PyTorch/CUDA versions, and Google Colab's base image changes over time. The installation cells below preserve the original approach, but a fresh runtime may require environment adjustments. Do not assume the notebook is currently reproducible without testing it end-to-end.

## Before you start
- Use photographs of the same mostly static scene, with substantial overlap between neighboring views.
- Store your images in a Google Drive folder.
- Use a Colab GPU runtime: **Runtime → Change runtime type → T4 GPU** (or another compatible GPU).
- Training consumes GPU time and Drive storage. The resulting PLY can be large.

## Workflow
1. Check the available GPU.
2. Mount Drive and load photographs.
3. Prepare the scene folder.
4. Run COLMAP to estimate camera poses and sparse geometry.
5. Install the 3DGS dependencies.
6. Train and export the PLY scene.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive and configure paths

Put your input images in a Drive folder. Edit `IMAGES_DIR` below if your folder is somewhere else. Supported extensions are JPG, JPEG, and PNG.

The scene and output folders are created under `/content` and Google Drive respectively; they do not depend on any personal account name.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import shutil

# EDIT THIS: Drive folder containing the input photographs.
IMAGES_DIR = "/content/drive/MyDrive/images"

# Scene workspace and output folder.
DATA = "/content/data/myscene"
OUTPUT_DIR = "/content/drive/MyDrive/gs_output/myscene"

image_dir = Path(IMAGES_DIR)
if not image_dir.is_dir():
    raise FileNotFoundError(
        f"Input folder not found: {IMAGES_DIR}. Update IMAGES_DIR to your Drive image folder."
    )

valid_exts = {".jpg", ".jpeg", ".png"}
imgs = sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts)
if not imgs:
    raise RuntimeError(f"No JPG/JPEG/PNG images found in {IMAGES_DIR}")

print(f"{len(imgs)} images found.")
print("Example filenames:", [p.name for p in imgs[:3]])
print("Scene folder:", DATA)
print("Output folder:", OUTPUT_DIR)


## 3. Download the 3D Gaussian Splatting implementation

In [ ]:
%cd /content
if not os.path.isdir("/content/gaussian-splatting"):
    !git clone --recursive https://github.com/camenduru/gaussian-splatting
%cd /content/gaussian-splatting
!ls


## 4. Copy images into the scene folder

In [ ]:
from pathlib import Path
import shutil

scene_input = Path(DATA) / "input"
scene_input.mkdir(parents=True, exist_ok=True)

# Start with a clean input directory so old images are not accidentally mixed in.
for old_file in scene_input.iterdir():
    if old_file.is_file():
        old_file.unlink()

for image_path in imgs:
    shutil.copy2(image_path, scene_input / image_path.name)

print(f"Copied {len(imgs)} images to {scene_input}")


## 5. Install COLMAP and prepare the scene

COLMAP estimates camera poses and sparse geometry from overlapping images. If registration fails, check image sharpness, overlap, texture, and whether the scene changed during capture.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap

%cd /content/gaussian-splatting
if not os.path.exists("convert.py"):
    !wget -q https://raw.githubusercontent.com/graphdeco-inria/gaussian-splatting/main/convert.py

!QT_QPA_PLATFORM=offscreen python3 convert.py -s {DATA} --no_gpu


In [ ]:
from pathlib import Path

sparse_dir = Path(DATA) / "sparse" / "0"
points3d = sparse_dir / "points3D.bin"
if not points3d.exists():
    raise RuntimeError(
        "COLMAP did not produce sparse/0/points3D.bin. "
        "Check the logs and input images before continuing."
    )

registered_dir = Path(DATA) / "images"
registered_count = len([p for p in registered_dir.iterdir() if p.is_file()]) if registered_dir.exists() else 0
print(f"Registered images copied by COLMAP: {registered_count} of {len(imgs)} input images")
print("Sparse reconstruction files:", sorted(p.name for p in sparse_dir.iterdir()))


## 6. Install the legacy training environment

The original run used Python 3.7 and PyTorch 1.12.1 with CUDA 11.6 wheels, plus a CUDA 11.8 toolkit installation. These versions are old, and this combination may not work in a current Colab runtime without changes.

**Run this section only if you need to reproduce the original environment.** Installing another Python/PyTorch/CUDA stack into a preconfigured Colab runtime can break other packages or require a runtime restart.

In [ ]:
# Legacy setup from the original experiment.
# This is intentionally kept explicit rather than silently installing a different stack.
!wget -O /content/miniconda-py37.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x /content/miniconda-py37.sh
!bash /content/miniconda-py37.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append("/usr/local/lib/python3.7/site-packages")
!python --version


### Install CUDA toolkit 11.8 (legacy setup)

In [ ]:
!wget -nc https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page

import os
os.environ["PATH"] = "/usr/local/cuda-11.8/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
!nvcc --version


### Install PyTorch and the 3DGS extensions

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn


## 7. Train the scene

The command below trains for 30,000 iterations and saves Gaussian point clouds and checkpoints at the configured iterations. Adjust the values if you have limited GPU time or memory.

In [ ]:
%cd /content/gaussian-splatting
os.makedirs(OUTPUT_DIR, exist_ok=True)

!python train.py -s {DATA} \
    -m {OUTPUT_DIR} \
    --iterations 30000 \
    --save_iterations 7000 15000 30000 \
    --checkpoint_iterations 7000 15000


## 8. Locate the exported PLY

In [ ]:
from pathlib import Path

PLY = Path(OUTPUT_DIR) / "point_cloud" / "iteration_30000" / "point_cloud.ply"
if PLY.exists():
    print(f"PLY saved: {PLY}")
    print(f"Size: {PLY.stat().st_size / (1024**2):.1f} MiB")
else:
    print(f"PLY not found at {PLY}. Check the training log and output directory.")


## Troubleshooting

- **No images found:** check `IMAGES_DIR`, file extensions, and Drive mount status.
- **COLMAP fails:** use sharper images with greater overlap and visible texture; avoid moving the object or changing lighting between shots.
- **CUDA extension compilation fails:** the installed PyTorch, CUDA toolkit, compiler, and Colab runtime may be incompatible. Use a version-matched environment and consult the upstream 3DGS repository.
- **Out of memory:** reduce input image resolution or use a GPU with more memory.
- **No PLY appears:** verify that training completed and that `OUTPUT_DIR` is correct.

## Credits and licensing

This notebook relies on an external 3D Gaussian Splatting implementation and COLMAP. Review the upstream repositories for current installation instructions, license terms, and citation requirements before redistribution or commercial use:

- [3D Gaussian Splatting implementation](https://github.com/graphdeco-inria/gaussian-splatting)
- [COLMAP](https://colmap.github.io/)
- [Original 3D Gaussian Splatting paper and project page](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)

This notebook is an educational workflow, not a new implementation of the 3DGS algorithm. The reported results from the original run should be treated as experiment-specific and are not a benchmark.
